# 📊 Amazon Reviews Dataset Preparation

We'll begin by preprocessing the Amazon Reviews dataset from Kaggle.

In [1]:
import pandas as pd

## Explore the Data

The dataset has 3 columns:
- **label**: Polarity of the review (Negative = 1, Positive = 2)
- **title**: Title of the review
- **review**: The review itself

In [2]:
train_df = pd.read_csv('data/train.csv', header=0, names=['label', 'title', 'review'])
test_df = pd.read_csv('data/test.csv', header=0, names=['label', 'title', 'review'])

In [3]:
train_df.head()

,label,title,review
0,2,The best soundtrack ever to anything.,I'm reading a lot of reviews saying that this ...
1,2,Amazing!,This soundtrack is my favorite music of all ti...
2,2,Excellent Soundtrack,I truly like this soundtrack and I enjoy video...
3,2,"Remember, Pull Your Jaw Off The Floor After He...","If you've played the game, you know how divine..."
4,2,an absolute masterpiece,I am quite sure any of you actually taking the...


In [4]:
train_df.shape

(3599999, 3)

In [5]:
test_df.head()

,label,title,review
0,2,One of the best game music soundtracks - for a...,Despite the fact that I have only played a sma...
1,1,Batteries died within a year ...,I bought this charger in Jul 2003 and it worke...
2,2,"works fine, but Maha Energy is better",Check out Maha Energy's website. Their Powerex...
3,2,Great for the non-audiophile,Reviewed quite a bit of the combo players and ...
4,1,DVD Player crapped out after one year,I also began having the incorrect disc problem...


In [6]:
test_df.shape

(399999, 3)

## Merge the Train and Test Sets

We'll merge both the train and test sets then create our own train/test split later on.

In [7]:
full_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)\
            .sample(frac=1, random_state=1, ignore_index=True)

In [8]:
full_df.head()

,label,title,review
0,1,Print too small,"I love the look of the bible, but the print is..."
1,1,Smelly,I thought the Diaper Champ was the best thing ...
2,2,Absolutely the best CD I've bought in years!,Buy it. You wont be able to stop listening to ...
3,1,Don't Buy It Unless U Get it as a Gift,I got the Avent sterilizer as a shower present...
4,1,15 minutes are missing from this DVD,"Having seen this film in the theater, I was ve..."


In [9]:
full_df.shape

(3999998, 3)

## Drop the ```title``` column
The pipelines will analyze the sentiment of the reviews (not the titles) so we can drop the ```title``` column.

In [10]:
full_df = full_df.drop(columns='title')
full_df.head()

,label,review
0,1,"I love the look of the bible, but the print is..."
1,1,I thought the Diaper Champ was the best thing ...
2,2,Buy it. You wont be able to stop listening to ...
3,1,I got the Avent sterilizer as a shower present...
4,1,"Having seen this film in the theater, I was ve..."


## Standardize the labels
- Original labels (1=negative, 2=positive)
- New labels (0=negative, 1=positive)

Let's provide the predictions in the typical format to improve user experience (UIX). 

In [11]:
def new_labels(x):
    if x == 1:
        return 0
    elif x == 2:
        return 1

full_df['new_label'] = full_df['label'].apply(new_labels)

In [12]:
full_df.head()

,label,review,new_label
0,1,"I love the look of the bible, but the print is...",0
1,1,I thought the Diaper Champ was the best thing ...,0
2,2,Buy it. You wont be able to stop listening to ...,1
3,1,I got the Avent sterilizer as a shower present...,0
4,1,"Having seen this film in the theater, I was ve...",0


## Drop the ```label``` column
Let's drop the old ```label``` column.

In [13]:
full_df = full_df.drop(columns='label')
full_df.head()

,review,new_label
0,"I love the look of the bible, but the print is...",0
1,I thought the Diaper Champ was the best thing ...,0
2,Buy it. You wont be able to stop listening to ...,1
3,I got the Avent sterilizer as a shower present...,0
4,"Having seen this film in the theater, I was ve...",0


In [14]:
full_df.rename(columns={'new_label':'label'}, inplace=True)

## Missing Values

In [15]:
full_df.isnull().value_counts()

review  label
False   False    3999998
Name: count, dtype: int64

## Duplicate Values

In [16]:
full_df.duplicated().value_counts()

False    3993773
True        6225
Name: count, dtype: int64

### Remove Duplicates

In [17]:
full_df = full_df.drop_duplicates()

In [18]:
full_df.duplicated().value_counts()

False    3993773
Name: count, dtype: int64

## Class Imbalances

In [19]:
full_df['label'].value_counts()

label
1    1998347
0    1995426
Name: count, dtype: int64

### Fix Class Imbalances

In [20]:
positives = full_df[full_df['label'] == 1]\
                .sample(frac=1, random_state=1, ignore_index=True)\
                .loc[:1995425, ['review','label']]

negatives = full_df[full_df['label'] == 0]

# Make sure same number of classes
negatives.shape == positives.shape

True

In [21]:
final_train_df = pd.concat([positives, negatives], axis=0, ignore_index=True)\
                   .sample(frac=1, random_state=1, ignore_index=True)

In [22]:
final_train_df.shape

(3990852, 2)

## 🎉 Save the Dataset as CSV

Save all your data preprocessing work!

In [23]:
final_train_df.to_csv('data/full_cleaned_dataset.csv', index=False)